<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File 
  </span>
</div>

Wrapped version of demoFile_raw.ipynb

It uses definitions for repeated used functions

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260723", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [2]:
# ==========================================================================================================
# Robot Initialization
# ==========================================================================================================
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
if (robot != None):
    model = robot.model()
    
time.sleep(1)


# ==========================================================================================================
# Gripper Initialization
# Gripper Setting (Need to hard check of gripper ping test)
# ==========================================================================================================
gripper = ysw.Gripper()
gripper.connect_hardCheck(
    device_name=ysw.GRIPPER_DEVICE_NAME, 
    protocol_version=ysw.DXL_PROTOCOL_VERSION, 
    baudrate=ysw.DXL_BAUDRATE,
    retry_interval=1.0,
    max_retry=5
)


# ==========================================================================================================
# Depth Camera Initialization
# D435i:  Head
# D405_R: Right Wrist
# D405_L: Left Wrist
# ==========================================================================================================
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()
d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()
d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!


[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************



*********************** [Gripper Hard Connection Starts] ***********************

[Gripper] Trying to connect gripper... attempt=1
************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



[Gripper] connect=True, ping_test=True

********************** [Gripper Hard Connection 

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Tuning Parameters
</span>

In [4]:
position_control_minimum_time = 8.5      # [sec]

# Move to Table
vision_record_duration_tableDistance = 5.0    # [sec]
offset_table_distance = 0.37                  # [m]
vision_record_duration_checkBox = 5.0         # [sec]
vision_record_duration_checkObject = 5.0      # [sec]
box_minimum_area = 8000                       # [pixel^2]
box_maximum_distance = 1.2                    # [m]
object_minimum_area = 50                      # [pixel^2]
object_maximum_area = 6000                    # [pixel^2]
object_maximum_distance = 1.2                 # [m]

# minimum_time_scaler = 0.5                # For short movement
grasp_current_obj = 0.1      # [A]
object_floating_z = 0.125     # [m]
offset_gripper_open = 0.015  # [m]
manual_offset_downToGrasp = -0.008       # [m] for z-direction offset when grasp


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [4]:
desired_pos = ysw.PACK_POS  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************

[ERROR] Failed to conduct [move_position_22dof].finish_code = FinishCode.ControlManagerFault


False

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [5 seconds]
</span>

In [4]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

record_duration = vision_record_duration_tableDistance      # [sec]
distance_list = []

d435i.startPipeline()

try:
    start_time = time.time()
    
    while True:
        elapsed_time = time.time() - start_time
        
        if elapsed_time >= record_duration:
            break
        
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        h, w, _ = color_image.shape
        
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        distance_list.append(distance)
        print(f"Table Distance: {distance:.3f} [m]",)

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
    

# Average Distance
distance_avg = np.mean(distance_list)
print(f"\nTable Average Distance: {distance_avg} [m]")

moved_distance_to_table = distance_avg - offset_table_distance
print(f"moved_distance_to_table: {moved_distance_to_table} [m]")

[INFO] [Head D435i] Pipeline started.
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.647 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.637 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.640 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.639 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.638 [m]
Table Distance: 0.637 [m]


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [5]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [9]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_table,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)

distance=0.000[m], speed=0.016[m/s], pos_error=1.261[m], yaw_error=0.00[deg]
distance=0.010[m], speed=0.144[m/s], pos_error=1.251[m], yaw_error=-0.02[deg]
distance=0.051[m], speed=0.250[m/s], pos_error=1.209[m], yaw_error=-0.01[deg]
distance=0.103[m], speed=0.250[m/s], pos_error=1.157[m], yaw_error=-0.11[deg]
distance=0.156[m], speed=0.250[m/s], pos_error=1.105[m], yaw_error=0.08[deg]
distance=0.208[m], speed=0.250[m/s], pos_error=1.053[m], yaw_error=0.09[deg]
distance=0.260[m], speed=0.250[m/s], pos_error=1.001[m], yaw_error=-0.00[deg]
distance=0.312[m], speed=0.250[m/s], pos_error=0.949[m], yaw_error=-0.01[deg]
distance=0.365[m], speed=0.250[m/s], pos_error=0.896[m], yaw_error=-0.03[deg]
distance=0.417[m], speed=0.250[m/s], pos_error=0.844[m], yaw_error=-0.01[deg]
distance=0.469[m], speed=0.250[m/s], pos_error=0.792[m], yaw_error=-0.01[deg]
distance=0.521[m], speed=0.250[m/s], pos_error=0.739[m], yaw_error=-0.02[deg]
distance=0.573[m], speed=0.250[m/s], pos_error=0.688[m], yaw_error=

True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black & Red] Boxes [5 seconds]
</span>

In [ ]:
def check_black_and_red_boxes_and_set_grasp_arm(
    d435i,
    record_duration,
    box_minimum_area,
    box_maximum_distance,
    pixel_roi_x=None,
    pixel_roi_y=None,
    verbose=True,
):
    """
    Detect black/red boxes for a certain duration and decide grasp arm.

    Return:
        result = {
            "blackBox_samples": blackBox_samples,
            "redBox_samples": redBox_samples,
            "blackBox_sample_avg": blackBox_sample_avg,
            "redBox_sample_avg": redBox_sample_avg,
            "grasp_arm_blackObj": grasp_arm_blackObj,
            "grasp_arm_redObj": grasp_arm_redObj,
        }
    """

    # ============================================================
    # Get the Information about Black & Red Boxes
    # ============================================================
    blackBox_samples = []
    redBox_samples = []

    rv = d435i.startPipeline()
    if rv is False:
        raise RuntimeError("[D435i] Failed to start pipeline.")

    try:
        start_time = time.time()

        while True:
            elapsed_time = time.time() - start_time

            if elapsed_time >= record_duration:
                break

            color_frame, depth_frame = d435i.getRawFrames()

            if color_frame is None or depth_frame is None:
                continue

            color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

            if color_image is None or depth_image is None:
                continue

            # Detection용 원본 이미지 보존
            color_image_raw = color_image.copy()

            detected_blackBoxes, color_image_forBlackBox, mask_blackBox = d435i.detectColorObjectContour3D(
                color_image=color_image_raw.copy(),
                depth_frame=depth_frame,
                mask_list=ysw.HSV_MASK_BLACK_BOX,
                minimum_area=box_minimum_area,
                maximum_depth=box_maximum_distance,
                pixel_roi_x=pixel_roi_x,
                pixel_roi_y=pixel_roi_y,
            )

            detected_redBoxes, color_image_forRedBox, mask_redBox = d435i.detectColorObjectContour3D(
                color_image=color_image_raw.copy(),
                depth_frame=depth_frame,
                mask_list=ysw.HSV_MASK_RED_BOX,
                minimum_area=box_minimum_area,
                maximum_depth=box_maximum_distance,
                pixel_roi_x=pixel_roi_x,
                pixel_roi_y=pixel_roi_y,
            )

            if len(detected_blackBoxes) > 0:
                largest_blackBox = max(
                    detected_blackBoxes,
                    key=lambda obj: obj["area_2d"]
                )

                blackBox_samples.append({
                    "area_2d":      float(largest_blackBox["area_2d"]),
                    "pixel_center": np.array(largest_blackBox["pixel_center"], dtype=np.float64),
                    "center_3d":    np.array(largest_blackBox["center_3d"], dtype=np.float64),
                    "bbox_3d_size": np.array(largest_blackBox["bbox_3d_size"], dtype=np.float64),
                    "bbox_3d_min":  np.array(largest_blackBox["bbox_3d_min"], dtype=np.float64),
                    "bbox_3d_max":  np.array(largest_blackBox["bbox_3d_max"], dtype=np.float64),
                })

            if len(detected_redBoxes) > 0:
                largest_redBox = max(
                    detected_redBoxes,
                    key=lambda obj: obj["area_2d"]
                )

                redBox_samples.append({
                    "area_2d":      float(largest_redBox["area_2d"]),
                    "pixel_center": np.array(largest_redBox["pixel_center"], dtype=np.float64),
                    "center_3d":    np.array(largest_redBox["center_3d"], dtype=np.float64),
                    "bbox_3d_size": np.array(largest_redBox["bbox_3d_size"], dtype=np.float64),
                    "bbox_3d_min":  np.array(largest_redBox["bbox_3d_min"], dtype=np.float64),
                    "bbox_3d_max":  np.array(largest_redBox["bbox_3d_max"], dtype=np.float64),
                })

    finally:
        d435i.stopPipeline()
        d435i.destroyAllWindows()

    # =========================
    # Average Box
    # =========================
    blackBox_sample_avg = None
    redBox_sample_avg = None

    if len(blackBox_samples) > 0:
        blackBox_sample_avg = ysw.average_detectColorObjectContour3D(blackBox_samples)

    if len(redBox_samples) > 0:
        redBox_sample_avg = ysw.average_detectColorObjectContour3D(redBox_samples)

    # =========================
    # Set grasp arm
    # =========================
    grasp_arm_blackObj = None
    grasp_arm_redObj = None

    if blackBox_sample_avg is not None:
        if blackBox_sample_avg["center_3d_avg"][0] > 0:
            grasp_arm_blackObj = "right"
        else:
            grasp_arm_blackObj = "left"

    if redBox_sample_avg is not None:
        if redBox_sample_avg["center_3d_avg"][0] > 0:
            grasp_arm_redObj = "right"
        else:
            grasp_arm_redObj = "left"

    # =========================
    # Print Result
    # =========================
    if verbose:
        print("\n========== Black & Red box Detection Result ==========")

        if blackBox_sample_avg is None:
            print("No Black Box detected.")
        else:
            print("\n[Black Box]")
            print(f"num_samples             : {blackBox_sample_avg['num_samples']}")
            print(f"area_2d_avg             : {blackBox_sample_avg['area_2d_avg']:.1f}")
            print(f"area_2d_median          : {blackBox_sample_avg['area_2d_median']:.1f}")
            print(f"center_3d_avg           : {np.round(blackBox_sample_avg['center_3d_avg'], 4)} m")
            print(f"center_3d_median        : {np.round(blackBox_sample_avg['center_3d_median'], 4)} m")
            print(f"bbox_3d_size_avg        : {np.round(blackBox_sample_avg['bbox_3d_size_avg'], 4)} m")
            print(f"bbox_3d_size_median     : {np.round(blackBox_sample_avg['bbox_3d_size_median'], 4)} m")

        if redBox_sample_avg is None:
            print("No Red Box detected.")
        else:
            print("\n[Red Box]")
            print(f"num_samples             : {redBox_sample_avg['num_samples']}")
            print(f"area_2d_avg             : {redBox_sample_avg['area_2d_avg']:.1f}")
            print(f"area_2d_median          : {redBox_sample_avg['area_2d_median']:.1f}")
            print(f"center_3d_avg           : {np.round(redBox_sample_avg['center_3d_avg'], 4)} m")
            print(f"center_3d_median        : {np.round(redBox_sample_avg['center_3d_median'], 4)} m")
            print(f"bbox_3d_size_avg        : {np.round(redBox_sample_avg['bbox_3d_size_avg'], 4)} m")
            print(f"bbox_3d_size_median     : {np.round(redBox_sample_avg['bbox_3d_size_median'], 4)} m")

        print(f"\ngrasp_arm_blackObj: {grasp_arm_blackObj}")
        print(f"grasp_arm_redObj: {grasp_arm_redObj}")

    result = {
        "blackBox_samples": blackBox_samples,
        "redBox_samples": redBox_samples,
        "blackBox_sample_avg": blackBox_sample_avg,
        "redBox_sample_avg": redBox_sample_avg,
        "grasp_arm_blackObj": grasp_arm_blackObj,
        "grasp_arm_redObj": grasp_arm_redObj,
    }

    return result

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None
pixel_roi_y = [150, height]

result_boxCheck = check_black_and_red_boxes_and_set_grasp_arm(
    d435i=d435i,
    record_duration=vision_record_duration_checkObject,
    box_minimum_area=box_minimum_area,
    box_maximum_distance=box_maximum_distance,
    pixel_roi_x=pixel_roi_x,
    pixel_roi_y=pixel_roi_y,
    verbose=True,
)

blackBox_sample_avg = result_boxCheck["blackBox_sample_avg"]
redBox_sample_avg = result_boxCheck["redBox_sample_avg"]

grasp_arm_blackObj = result_boxCheck["grasp_arm_blackObj"]
grasp_arm_redObj = result_boxCheck["grasp_arm_redObj"]

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black Leftmost & Red Rightmost] Object [5 seconds]
</span>

In [ ]:
def check_leftmost_black_and_rightmost_red_objects(
    d435i,
    record_duration,
    object_minimum_area,
    object_maximum_area,
    maximum_depth,
    pixel_roi_x=None,
    pixel_roi_y=None,
    verbose=True,
):
    """
    Detect black/red objects for a certain duration.

    - Black object : use leftmost object
    - Red object   : use rightmost object

    Return:
        result = {
            "leftmost_blackObj_samples": ...,
            "rightmost_redObj_samples": ...,
            "leftmost_blackObj_sample_avg": ...,
            "rightmost_redObj_sample_avg": ...,
        }
    """

    leftmost_blackObj_samples = []
    rightmost_redObj_samples = []

    rv = d435i.startPipeline()
    if rv is False:
        raise RuntimeError("[D435i] Failed to start pipeline.")

    try:
        start_time = time.time()

        while True:
            elapsed_time = time.time() - start_time

            if elapsed_time >= record_duration:
                break

            color_frame, depth_frame = d435i.getRawFrames()

            if color_frame is None or depth_frame is None:
                continue

            color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

            if color_image is None or depth_image is None:
                continue

            # Detection용 원본 이미지 보존
            color_image_raw = color_image.copy()

            detected_blackObjects, color_image_forBlackObj, mask_blackObj = d435i.detectColorObjectContour3D(
                color_image=color_image_raw.copy(),
                depth_frame=depth_frame,
                mask_list=ysw.HSV_MASK_BLACK_BOX,
                minimum_area=object_minimum_area,
                maximum_area=object_maximum_area,
                maximum_depth=maximum_depth,
                pixel_roi_x=pixel_roi_x,
                pixel_roi_y=pixel_roi_y,
            )

            detected_redObjects, color_image_forRedObj, mask_redObj = d435i.detectColorObjectContour3D(
                color_image=color_image_raw.copy(),
                depth_frame=depth_frame,
                mask_list=ysw.HSV_MASK_RED,
                minimum_area=object_minimum_area,
                maximum_area=object_maximum_area,
                maximum_depth=maximum_depth,
                pixel_roi_x=pixel_roi_x,
                pixel_roi_y=pixel_roi_y,
            )

            # =========================
            # Leftmost black object
            # =========================
            if len(detected_blackObjects) > 0:
                # RealSense/D435i 기준:
                # X+ = image right
                # 따라서 center_3d[0]이 작을수록 왼쪽
                blackObj_list_fromLeft = sorted(
                    detected_blackObjects,
                    key=lambda obj: obj["center_3d"][0],
                    reverse=False
                )

                obj = blackObj_list_fromLeft[0]

                leftmost_blackObj_samples.append({
                    "area_2d":      float(obj["area_2d"]),
                    "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                    "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                    "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                    "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                    "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
                })

            # =========================
            # Rightmost red object
            # =========================
            if len(detected_redObjects) > 0:
                # RealSense/D435i 기준:
                # X+ = image right
                # 따라서 center_3d[0]이 클수록 오른쪽
                redObj_list_fromLeft = sorted(
                    detected_redObjects,
                    key=lambda obj: obj["center_3d"][0],
                    reverse=True
                )

                obj = redObj_list_fromLeft[0]

                rightmost_redObj_samples.append({
                    "area_2d":      float(obj["area_2d"]),
                    "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                    "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                    "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                    "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                    "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
                })

    finally:
        d435i.stopPipeline()
        d435i.destroyAllWindows()

    # =========================
    # Average objects
    # =========================
    leftmost_blackObj_sample_avg = None
    rightmost_redObj_sample_avg = None

    if len(leftmost_blackObj_samples) > 0:
        leftmost_blackObj_sample_avg = ysw.average_detectColorObjectContour3D(
            leftmost_blackObj_samples
        )

    if len(rightmost_redObj_samples) > 0:
        rightmost_redObj_sample_avg = ysw.average_detectColorObjectContour3D(
            rightmost_redObj_samples
        )

    # =========================
    # Print result
    # =========================
    if verbose:
        print("\n========== Detection Result ==========")
        print("Black: leftmost object / Red: rightmost object\n")

        if leftmost_blackObj_sample_avg is None:
            print("No Black Object detected.")
        else:
            print("\n[Leftmost Black Object]")
            print(f"num_samples             : {leftmost_blackObj_sample_avg['num_samples']}")
            print(f"area_2d_avg             : {leftmost_blackObj_sample_avg['area_2d_avg']:.1f}")
            print(f"area_2d_median          : {leftmost_blackObj_sample_avg['area_2d_median']:.1f}")
            print(f"center_3d_avg           : {np.round(leftmost_blackObj_sample_avg['center_3d_avg'], 4)} m")
            print(f"center_3d_median        : {np.round(leftmost_blackObj_sample_avg['center_3d_median'], 4)} m")
            print(f"bbox_3d_size_avg        : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_avg'], 4)} m")
            print(f"bbox_3d_size_median     : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_median'], 4)} m")

        if rightmost_redObj_sample_avg is None:
            print("No Red Object detected.")
        else:
            print("\n[Rightmost Red Object]")
            print(f"num_samples             : {rightmost_redObj_sample_avg['num_samples']}")
            print(f"area_2d_avg             : {rightmost_redObj_sample_avg['area_2d_avg']:.1f}")
            print(f"area_2d_median          : {rightmost_redObj_sample_avg['area_2d_median']:.1f}")
            print(f"center_3d_avg           : {np.round(rightmost_redObj_sample_avg['center_3d_avg'], 4)} m")
            print(f"center_3d_median        : {np.round(rightmost_redObj_sample_avg['center_3d_median'], 4)} m")
            print(f"bbox_3d_size_avg        : {np.round(rightmost_redObj_sample_avg['bbox_3d_size_avg'], 4)} m")
            print(f"bbox_3d_size_median     : {np.round(rightmost_redObj_sample_avg['bbox_3d_size_median'], 4)} m")

    result = {
        "leftmost_blackObj_samples": leftmost_blackObj_samples,
        "rightmost_redObj_samples": rightmost_redObj_samples,
        "leftmost_blackObj_sample_avg": leftmost_blackObj_sample_avg,
        "rightmost_redObj_sample_avg": rightmost_redObj_sample_avg,
    }

    return result

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Detection Result ==========
Only the Leftmost object is used


[Leftmost Black Object]
num_samples             : 132
area_2d_avg             : 499.8
area_2d_median          : 502.5
center_3d_avg           : [ 0.0838 -0.0113  0.804 ] m
center_3d_median        : [ 0.0848 -0.0124  0.805 ] m
bbox_3d_size_avg        : [0.0293 0.0349 0.0286] m
bbox_3d_size_median     : [0.0297 0.0353 0.031 ] m

[Rightmost Red Object]
num_samples             : 130
area_2d_avg             : 699.3
area_2d_median          : 643.2
center_3d_avg           : [0.0048 0.0937 0.688 ] m
center_3d_median        : [0.0104 0.0911 0.688 ] m
bbox_3d_size_avg        : [0.0313 0.0316 0.0303] m
bbox_3d_size_median     : [0.0263 0.0324 0.03  ] m


In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None
pixel_roi_y = [150, height]

result_objectCheck = check_leftmost_black_and_rightmost_red_objects(
    d435i=d435i,
    record_duration=vision_record_duration_checkBox,
    object_minimum_area=object_minimum_area,
    object_maximum_area=object_maximum_area,
    maximum_depth=box_maximum_distance,
    pixel_roi_x=pixel_roi_x,
    pixel_roi_y=pixel_roi_y,
    verbose=True,
)

leftmost_blackObj_sample_avg = result_objectCheck["leftmost_blackObj_sample_avg"]
rightmost_redObj_sample_avg = result_objectCheck["rightmost_redObj_sample_avg"]

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
IK for [Black-Leftmost & Red-Rightmost] Objects
</span>

In [ ]:
def compute_IK_for_object_and_box(
    robot,
    leftmost_blackObj_sample_avg,
    rightmost_redObj_sample_avg,
    blackBox_sample_avg,
    redBox_sample_avg,
    grasp_arm_blackObj,
    grasp_arm_redObj,
    manual_offset_downToGrasp=-0.008,   # can be manually changed
    pregrasp_z_offset=+0.1,             # can be manually changed
    use_median=False,
    gripper_z_offset=ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER,
    verbose=True,
):
    """
    Compute target transforms for:
        - black object pre-grasp / grasp
        - red object pre-grasp / grasp
        - black collecting box
        - red collecting box

    Frame notation:
        {V}: vision frame
        {H}: link_head_2 frame
        {S}: base frame
        {M}: object goal frame
        {M1}: object goal frame before opening gripper
        {M2}: object goal frame after opening gripper
        {N}: collecting box goal frame
        {C}: ee_finger_l1/r1 frame
        {D}: final fingertip frame, same orientation as {C}

    Args:
        use_median:
            False -> use center_3d_avg / bbox_3d_size_avg
            True  -> use center_3d_median / bbox_3d_size_median

    Return:
        result dict
    """

    center_key = "center_3d_median" if use_median else "center_3d_avg"
    size_key = "bbox_3d_size_median" if use_median else "bbox_3d_size_avg"

    # ------------------------------------------------------------
    # 1. Get object / box positions in vision frame {V}
    # ------------------------------------------------------------
    leftmost_blackObj_pos = np.asarray(
        leftmost_blackObj_sample_avg[center_key],
        dtype=np.float64
    )
    leftmost_blackObj_3Dsize = np.asarray(
        leftmost_blackObj_sample_avg[size_key],
        dtype=np.float64
    )

    rightmost_redObj_pos = np.asarray(
        rightmost_redObj_sample_avg[center_key],
        dtype=np.float64
    )
    rightmost_redObj_3Dsize = np.asarray(
        rightmost_redObj_sample_avg[size_key],
        dtype=np.float64
    )

    blackBox_pos = np.asarray(
        blackBox_sample_avg[center_key],
        dtype=np.float64
    )
    blackBox_3Dsize = np.asarray(
        blackBox_sample_avg[size_key],
        dtype=np.float64
    )

    redBox_pos = np.asarray(
        redBox_sample_avg[center_key],
        dtype=np.float64
    )
    redBox_3Dsize = np.asarray(
        redBox_sample_avg[size_key],
        dtype=np.float64
    )

    # ------------------------------------------------------------
    # 2. Convert the position based on {V} to the position based on {H}
    # ------------------------------------------------------------
    P_HM_blackObj = ysw.convertPos_vision_to_linkhead2(leftmost_blackObj_pos)
    P_HM_redObj = ysw.convertPos_vision_to_linkhead2(rightmost_redObj_pos)

    P_HN_blackBox = ysw.convertPos_vision_to_linkhead2(blackBox_pos)
    P_HN_redBox = ysw.convertPos_vision_to_linkhead2(redBox_pos)

    # ------------------------------------------------------------
    # 3. Get T_SH: base -> link_head_2
    # ------------------------------------------------------------
    fk_SH = ysw.get_FK(robot, "base", "link_head_2")

    T_SH = np.asarray(fk_SH["T"], dtype=np.float64)
    R_SH = np.asarray(fk_SH["rotation"], dtype=np.float64)
    P_SH = np.asarray(fk_SH["position"], dtype=np.float64)

    # ------------------------------------------------------------
    # 4. Convert the position based on {H} to the position based on {S}
    # ------------------------------------------------------------
    def transform_point(T, P):
        P_ext = np.append(np.asarray(P, dtype=np.float64), 1.0)
        P_out = T @ P_ext
        return P_out[:3]

    P_SM_blackObj = transform_point(T_SH, P_HM_blackObj)
    P_SM_redObj = transform_point(T_SH, P_HM_redObj)

    P_SN_blackBox = transform_point(T_SH, P_HN_blackBox)
    P_SN_redBox = transform_point(T_SH, P_HN_redBox)

    # ------------------------------------------------------------
    # 5. Desired gripper orientation (Can be changed the angle)
    # ------------------------------------------------------------
    def get_desired_gripper_rotation(grasp_arm):
        if grasp_arm == "right":
            R1 = ysw.rotMat_z(90.0 * ysw.DEG2RAD)
            R2 = ysw.rotMat_x(-40.0 * ysw.DEG2RAD)
            return R1 @ R2

        elif grasp_arm == "left":
            R1 = ysw.rotMat_z(-90.0 * ysw.DEG2RAD)
            R2 = ysw.rotMat_x(40.0 * ysw.DEG2RAD)
            return R1 @ R2

        else:
            raise ValueError(f"Invalid grasp_arm: {grasp_arm}")

    R_SC_desired_blackObj = get_desired_gripper_rotation(grasp_arm_blackObj)
    R_SC_desired_redObj = get_desired_gripper_rotation(grasp_arm_redObj)

    # ------------------------------------------------------------
    # 6. Box target transforms
    # ------------------------------------------------------------
    R_SN_blackBox = R_SC_desired_blackObj
    T_SN_blackBox = ysw.make_HT(R_SN_blackBox, P_SN_blackBox)

    R_SN_redBox = R_SC_desired_redObj
    T_SN_redBox = ysw.make_HT(R_SN_redBox, P_SN_redBox)

    # ------------------------------------------------------------
    # 7. Object pre-grasp / grasp target transforms
    # ------------------------------------------------------------
    P_SM1_blackObj = P_SM_blackObj + np.array([0.0, 0.0, pregrasp_z_offset])
    R_SM1_blackObj = R_SC_desired_blackObj
    T_SM1_blackObj = ysw.make_HT(R_SM1_blackObj, P_SM1_blackObj)

    P_SM2_blackObj = P_SM_blackObj + np.array([0.0, 0.0, manual_offset_downToGrasp])
    R_SM2_blackObj = R_SC_desired_blackObj
    T_SM2_blackObj = ysw.make_HT(R_SM2_blackObj, P_SM2_blackObj)

    P_SM1_redObj = P_SM_redObj + np.array([0.0, 0.0, pregrasp_z_offset])
    R_SM1_redObj = R_SC_desired_redObj
    T_SM1_redObj = ysw.make_HT(R_SM1_redObj, P_SM1_redObj)

    P_SM2_redObj = P_SM_redObj + np.array([0.0, 0.0, manual_offset_downToGrasp])
    R_SM2_redObj = R_SC_desired_redObj
    T_SM2_redObj = ysw.make_HT(R_SM2_redObj, P_SM2_redObj)

    # ------------------------------------------------------------
    # 8. ee_finger frame {C} -> fingertip frame {D}
    # ------------------------------------------------------------
    P_CD = np.array([0.0, 0.0, gripper_z_offset], dtype=np.float64)
    T_CD = ysw.make_HT(np.eye(3), P_CD)
    T_DC = np.linalg.inv(T_CD)

    # Target of {C} = target of {D} * inv(T_CD)
    T_SC_target_before_open_gripper_blackObj = T_SM1_blackObj @ T_DC
    T_SC_target_after_open_gripper_blackObj = T_SM2_blackObj @ T_DC

    T_SC_target_before_open_gripper_redObj = T_SM1_redObj @ T_DC
    T_SC_target_after_open_gripper_redObj = T_SM2_redObj @ T_DC

    T_SC_target_for_blackBox = T_SN_blackBox @ T_DC
    T_SC_target_for_redBox = T_SN_redBox @ T_DC

    # ------------------------------------------------------------
    # 9. Print
    # ------------------------------------------------------------
    if verbose:
        print("======== For Black Object ========")
        print(f"Before open gripper: \n{T_SC_target_before_open_gripper_blackObj}\n")
        print(f"After open gripper: \n{T_SC_target_after_open_gripper_blackObj}\n")
        print(f"Grasping Arm: {grasp_arm_blackObj}\n")

        print("======== For Red Object ========")
        print(f"Before open gripper: \n{T_SC_target_before_open_gripper_redObj}\n")
        print(f"After open gripper: \n{T_SC_target_after_open_gripper_redObj}\n")
        print(f"Grasping Arm: {grasp_arm_redObj}\n")

        print("======== For Black Box ========")
        print(f"For Black Box: \n{T_SC_target_for_blackBox}\n")

        print("======== For Red Box ========")
        print(f"For Red Box: \n{T_SC_target_for_redBox}\n")

    # ------------------------------------------------------------
    # 10. Return all useful values
    # ------------------------------------------------------------
    result = {
        # vision frame values
        "leftmost_blackObj_pos_V": leftmost_blackObj_pos,
        "rightmost_redObj_pos_V": rightmost_redObj_pos,
        "blackBox_pos_V": blackBox_pos,
        "redBox_pos_V": redBox_pos,

        "leftmost_blackObj_3Dsize": leftmost_blackObj_3Dsize,
        "rightmost_redObj_3Dsize": rightmost_redObj_3Dsize,
        "blackBox_3Dsize": blackBox_3Dsize,
        "redBox_3Dsize": redBox_3Dsize,

        # head frame values
        "P_HM_blackObj": P_HM_blackObj,
        "P_HM_redObj": P_HM_redObj,
        "P_HN_blackBox": P_HN_blackBox,
        "P_HN_redBox": P_HN_redBox,

        # base frame values
        "T_SH": T_SH,
        "R_SH": R_SH,
        "P_SH": P_SH,

        "P_SM_blackObj": P_SM_blackObj,
        "P_SM_redObj": P_SM_redObj,
        "P_SN_blackBox": P_SN_blackBox,
        "P_SN_redBox": P_SN_redBox,

        # desired rotations
        "R_SC_desired_blackObj": R_SC_desired_blackObj,
        "R_SC_desired_redObj": R_SC_desired_redObj,

        # intermediate target transforms
        "T_SM1_blackObj": T_SM1_blackObj,
        "T_SM2_blackObj": T_SM2_blackObj,
        "T_SM1_redObj": T_SM1_redObj,
        "T_SM2_redObj": T_SM2_redObj,
        "T_SN_blackBox": T_SN_blackBox,
        "T_SN_redBox": T_SN_redBox,

        # fingertip offset
        "P_CD": P_CD,
        "T_CD": T_CD,

        # final ee_finger target transforms
        "T_SC_target_before_open_gripper_blackObj": T_SC_target_before_open_gripper_blackObj,
        "T_SC_target_after_open_gripper_blackObj": T_SC_target_after_open_gripper_blackObj,
        "T_SC_target_before_open_gripper_redObj": T_SC_target_before_open_gripper_redObj,
        "T_SC_target_after_open_gripper_redObj": T_SC_target_after_open_gripper_redObj,
        "T_SC_target_for_blackBox": T_SC_target_for_blackBox,
        "T_SC_target_for_redBox": T_SC_target_for_redBox,
    }

    return result

In [ ]:
IK_result = compute_IK_for_object_and_box(
    robot=robot,
    leftmost_blackObj_sample_avg=leftmost_blackObj_sample_avg,
    rightmost_redObj_sample_avg=rightmost_redObj_sample_avg,
    blackBox_sample_avg=blackBox_sample_avg,
    redBox_sample_avg=redBox_sample_avg,
    grasp_arm_blackObj=grasp_arm_blackObj,
    grasp_arm_redObj=grasp_arm_redObj,
    manual_offset_downToGrasp=manual_offset_downToGrasp,   # can be manually changed
    pregrasp_z_offset=+0.1,             # can be manually changed
    use_median=False,
    gripper_z_offset=ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER,
    verbose=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Pick [Black-Leftmost object] and Place to [Black Box]
</span>

In [ ]:
def get_target_ee(grasp_arm):
    if grasp_arm == "right":
        return "ee_finger_r1"
    elif grasp_arm == "left":
        return "ee_finger_l1"
    else:
        raise ValueError(f"Invalid grasp_arm: {grasp_arm}")
    

def get_target_gripper_id(grasp_arm):
    if grasp_arm == "right":
        return ysw.GRIPPER_ID_R
    elif grasp_arm == "left":
        return ysw.GRIPPER_ID_L
    else:
        raise ValueError(f"Invalid grasp_arm: {grasp_arm}")  
    
    
def set_fixed_joint_targets(robot):
    """
        Fix current torso_0 ~ torso_3 joint positions.
        It can be changed by USER
    """
    current_joint_position = ysw.getState(robot).position

    fixed_joint_targets = {
        "torso_0": current_joint_position[ysw.ALL_JOINT_INDEX["torso_0"]],
        "torso_1": current_joint_position[ysw.ALL_JOINT_INDEX["torso_1"]],
        "torso_2": current_joint_position[ysw.ALL_JOINT_INDEX["torso_2"]],
        "torso_3": current_joint_position[ysw.ALL_JOINT_INDEX["torso_3"]],
    }

    return fixed_joint_targets


def positionCtrl_ee_by_T(
    robot,
    T_SC_target,
    grasp_arm,
    minimum_time=8.5,
    control_hold_time=0.5,
    fix_torso=True,
    get_return=True,
):
    """
        Move selected ee finger frame to target transform.

        Args:
            T_SC_target:
                4x4 transform of ee_finger_l1/r1 target w.r.t. base frame.
            grasp_arm:
                "right" or "left"
    """

    T_SC_target = np.asarray(T_SC_target, dtype=np.float64)

    pos = T_SC_target[:3, 3].copy()
    ori = ysw.rotMat_to_rpy(T_SC_target[:3, :3])

    target_ee_link = get_target_ee(grasp_arm)

    cartesian_position_targets_sample = [
        {
            "reference_link": "base",
            "target_link": target_ee_link,
            "pos": pos,
            "rpy": ori,
        },
    ]

    if fix_torso:
        joint_targets = set_fixed_joint_targets(robot)
    else:
        joint_targets = None

    rv = ysw.positionCtrl_cartesian_wholeBody(
        robot,
        position_targets=cartesian_position_targets_sample,
        joint_targets=joint_targets,
        minimum_time=minimum_time,
        control_hold_time=control_hold_time,
        get_return=get_return,
    )
    return rv



def positionCtrl_by_pos_and_T_orientation(
    robot,
    pos,
    T_SC_orientation_source,
    grasp_arm,
    minimum_time=8.5,
    control_hold_time=0.5,
    fix_torso=True,
    get_return=True,
):
    """
        Move selected ee finger frame to given position,
        while maintaining orientation from T_SC_orientation_source.
    """

    T_SC_orientation_source = np.asarray(T_SC_orientation_source, dtype=np.float64)

    pos = np.asarray(pos, dtype=np.float64).copy()
    ori = ysw.rotMat_to_rpy(T_SC_orientation_source[:3, :3])

    target_link = get_target_ee(grasp_arm)

    cartesian_position_targets_sample = [
        {
            "reference_link": "base",
            "target_link": target_link,
            "pos": pos,
            "rpy": ori,
        },
    ]

    if fix_torso:
        joint_targets = set_fixed_joint_targets(robot)
    else:
        joint_targets = None

    rv = ysw.positionCtrl_cartesian_wholeBody(
        robot,
        position_targets=cartesian_position_targets_sample,
        joint_targets=joint_targets,
        minimum_time=minimum_time,
        control_hold_time=control_hold_time,
        get_return=get_return,
    )
    return rv
    
    

def compute_desired_gripper_open_pos_mm(
    object_3Dsize,
    offset_gripper_open,
    max_margin_mm=8.0,
):
    """
    Compute desired gripper opening width from object size.
    """

    object_3Dsize = np.asarray(object_3Dsize, dtype=np.float64)

    object_meanDim = np.mean(object_3Dsize)                 # [m]
    desiredGripperPos = object_meanDim + offset_gripper_open # [m]
    desiredGripperPos_mm = desiredGripperPos * 1000.0        # [mm]

    if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
        desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - max_margin_mm

    return desiredGripperPos_mm


def open_gripper_for_grasp(
    gripper,
    grasp_arm,
    desiredGripperPos_mm,
    current_ampere_mag=0.02,
    sleep_sec=1.0,
    disable_after_open=True,
):
    """
        Open selected gripper using current-based position control.
    """

    dxl_id = get_target_gripper_id(grasp_arm)

    gripper.initialize_setting(dxl_id=dxl_id, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(dxl_id, current_ampere_mag=current_ampere_mag, linear_pos=desiredGripperPos_mm)
    time.sleep(sleep_sec)

    if disable_after_open:
        gripper.disable(dxl_id)



def close_gripper_for_grasp(
    gripper,
    grasp_arm,
    grasp_current_obj,
    sleep_sec=1.0,
):
    """
        Close selected gripper using current control.
    """

    dxl_id = get_target_gripper_id(grasp_arm)

    gripper.initialize_setting(dxl_id=dxl_id, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.write_current_ampere(dxl_id, grasp_current_obj)
    time.sleep(sleep_sec)

***************************** [Robot State Starts] *****************************

RobotState_A(
  timestamp=datetime.datetime(1970, 1, 1, 13, 19, 39, 710040),
  system_stat=SystemStat(
    cpu_usage=0.0422,
    memory_usage=0.0682,
    uptime=4.8e+04,
    program_uptime=4.8e+04
  ),
  battery_state=BatteryState(
    voltage=57,
    current=3.5,
    level_percent=82
  ),
  power_states=[PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=0
  ), PowerState(
    state=PowerState.State.PowerOn,
    voltage=57
  )],
  emo_states=[EMOState(
    state=EMOState.State.Released
  )],
  joint_states=[JointState(
    time_since_last_update=datetime.timedelta(microseconds=2),
    is_ready=True,
    fet_state=JointState.FETState.On,
    run_state=JointState.RunState.ControlOn,
    init_state=JointState.InitializationState.Initialized,
    motor_type=1,
    mot


**************** [Cartesian Position Control(Whole-Body) Ends] *****************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********** [Gripper [DXL ID:1] Current-based Position Control Starts] **********

Desired Linear Position = 45.96210896014mm
[INFO] [DXL ID:1] SUCCESS: operation mode = 5
[INFO] SUCCESS: [DXL ID:1] Set Current [0.02 A]
[INFO] SUCCESS: [DXL ID:1] Set Goal Position [Tick]: signed=544, uint32=544

*********** [Gripper [DXL ID:1] Current-based Position Control Ends] ***********



********************* [Gripper [DXL ID:1] Disable Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled

********************* [Gripper [DXL ID:1] Disable is Done] *********************


In [ ]:
def pick_object_and_place_to_box(
    robot,
    gripper,
    grasp_arm,
    object_3Dsize,
    T_SC_target_before_open_gripper,
    T_SC_target_after_open_gripper,
    T_SC_target_for_box,
    offset_gripper_open,
    grasp_current_obj,
    object_floating_z,
    position_control_minimum_time,
    control_hold_time=0.5,
    fix_torso=True,
    verbose=True,
):
    """
    Pick one object and place it into the target box.

    Sequence:
        1. Move to pre-grasp pose
        2. Open gripper
        3. Move to grasp pose
        4. Close gripper
        5. Slightly float the object
        6. Move to box while keeping floating height
        7. Open gripper
    """

    # ======================================
    # Move to Pre-grasp pose
    # ======================================
    if verbose:
        print("\n# ======================================")
        print("# Move to Pre-grasp pose")
        print("# ======================================")

    rv = move_gripper_by_T(
        robot=robot,
        T_SC_target=T_SC_target_before_open_gripper,
        grasp_arm=grasp_arm,
        minimum_time=position_control_minimum_time,
        control_hold_time=control_hold_time,
        fix_torso=fix_torso,
        get_return=True,
    )

    if rv is False:
        print("[ERROR] Failed to move to pre-grasp pose.")
        return False

    # ======================================
    # Open Gripper
    # ======================================
    if verbose:
        print("\n# ======================================")
        print("# Open Gripper")
        print("# ======================================")

    desiredGripperPos_mm = compute_desired_gripper_open_pos_mm(
        object_3Dsize=object_3Dsize,
        offset_gripper_open=offset_gripper_open,
    )

    open_gripper_for_grasp(
        gripper=gripper,
        grasp_arm=grasp_arm,
        desiredGripperPos_mm=desiredGripperPos_mm,
        current_ampere_mag=0.02,
        sleep_sec=1.0,
        disable_after_open=True,
    )

    # ======================================
    # Move to grasp pose
    # ======================================
    if verbose:
        print("\n# ======================================")
        print("# Move to grasp pose")
        print("# ======================================")

    rv = move_gripper_by_T(
        robot=robot,
        T_SC_target=T_SC_target_after_open_gripper,
        grasp_arm=grasp_arm,
        minimum_time=position_control_minimum_time,
        control_hold_time=control_hold_time,
        fix_torso=fix_torso,
        get_return=True,
    )

    if rv is False:
        print("[ERROR] Failed to move to grasp pose.")
        return False

    # ======================================
    # Grasp
    # ======================================
    if verbose:
        print("\n# ======================================")
        print("# Grasp")
        print("# ======================================")

    close_gripper_for_grasp(
        gripper=gripper,
        grasp_arm=grasp_arm,
        grasp_current_obj=grasp_current_obj,
        sleep_sec=1.0,
    )

    # ========================================================
    # Slightly float the object
    # ========================================================
    if verbose:
        print("\n# ========================================================")
        print("# Slightly float the object")
        print("# ========================================================")

    T_SC_target_after_open_gripper = np.asarray(
        T_SC_target_after_open_gripper,
        dtype=np.float64
    )

    pos_after_open_gripper = T_SC_target_after_open_gripper[:3, 3].copy()
    pos_floating = pos_after_open_gripper + np.array([0.0, 0.0, object_floating_z])

    rv = move_gripper_by_pos_and_T_orientation(
        robot=robot,
        pos=pos_floating,
        T_SC_orientation_source=T_SC_target_after_open_gripper,
        grasp_arm=grasp_arm,
        minimum_time=position_control_minimum_time,
        control_hold_time=control_hold_time,
        fix_torso=fix_torso,
        get_return=True,
    )

    if rv is False:
        print("[ERROR] Failed to float the object.")
        return False

    # ========================================================
    # Move to Box
    # ========================================================
    if verbose:
        print("\n# ========================================================")
        print("# Move to Box")
        print("# ========================================================")

    T_SC_target_for_box = np.asarray(
        T_SC_target_for_box,
        dtype=np.float64
    )

    current_pos = pos_floating.copy()

    pos_box = T_SC_target_for_box[:3, 3].copy()
    pos_box[2] = current_pos[2]     # floating 이후 height 유지

    rv = move_gripper_by_pos_and_T_orientation(
        robot=robot,
        pos=pos_box,
        T_SC_orientation_source=T_SC_target_for_box,
        grasp_arm=grasp_arm,
        minimum_time=position_control_minimum_time,
        control_hold_time=control_hold_time,
        fix_torso=fix_torso,
        get_return=True,
    )

    if rv is False:
        print("[ERROR] Failed to move to box.")
        return False

    # ========================================================
    # Open Gripper
    # ========================================================
    if verbose:
        print("\n# ========================================================")
        print("# Open Gripper")
        print("# ========================================================")

    open_gripper_for_grasp(
        gripper=gripper,
        grasp_arm=grasp_arm,
        desiredGripperPos_mm=desiredGripperPos_mm,
        current_ampere_mag=0.02,
        sleep_sec=1.0,
        disable_after_open=True,
    )

    if verbose:
        print("[DONE] Pick and place sequence is done.")

    return True